# Byonoy Luminescence 96

The Luminescence 96 is a USB-HID plate reader from Byonoy that measures luminescence across a 96-well plate. It supports:

- [Luminescence](../../capabilities/luminescence) (full-plate, configurable integration time)

The hardware consists of a **base unit** (holds the plate) and a **reader unit** (detector, sits on top during measurement). PLR models both as resources so a robotic arm can move the reader unit on and off the base. Two hardware variants exist: the L96 (manual) and L96A (automate, with a preferred pickup location for robotic handling).

| Model | PLR Name | Factory function |
|---|---|---|
| L96 full setup | `ByonoyLuminescenceBaseUnit` + `ByonoyLuminescence96` | `byonoy_l96` |
| L96A full setup (automate) | `ByonoyLuminescenceBaseUnit` + `ByonoyLuminescence96` | `byonoy_l96a` |
| L96 reader unit only | `ByonoyLuminescence96` | `byonoy_l96_reader_unit` |
| L96A reader unit only | `ByonoyLuminescence96` | `byonoy_l96a_reader_unit` |
| L96 base unit only | `ByonoyLuminescenceBaseUnit` | `byonoy_l96_base_unit` |
| L96A base unit only | `ByonoyLuminescenceBaseUnit` | `byonoy_l96a_base_unit` |

- **Communication**: USB HID (VID `0x16D0` / PID `0x119B`)
- **Communication level**: Firmware

## Setup

Use `byonoy_l96a` (automate) or `byonoy_l96` (manual) to create the full setup (base unit + reader unit). The reader unit is both a `Resource` and a `Device`.

In [ ]:
from pylabrobot.byonoy import byonoy_l96a

base, reader = byonoy_l96a(name="l96a")
await reader.setup()

## Luminescence

The luminescence capability is exposed as `reader.luminescence`. For the full API, see [Luminescence](../../capabilities/luminescence).

Before reading, remove the reader unit from the base so a plate can be assigned, then place the reader unit back.

In [ ]:
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

# Remove reader unit so plate can be loaded
base.reader_unit_holder.unassign_child_resource(reader)

plate = Cor_96_wellplate_360ul_Fb(name="plate")
base.plate_holder.assign_child_resource(plate)

In [ ]:
results = await reader.luminescence.read_luminescence(plate, focal_height=13.0)

### Custom integration time

Use {class}`~pylabrobot.byonoy.luminescence_96.ByonoyLuminescence96Backend.LuminescenceParams` to set the integration time (in seconds, default 2).

In [ ]:
from pylabrobot.byonoy import ByonoyLuminescence96Backend

results = await reader.luminescence.read_luminescence(
    plate,
    focal_height=13.0,
    backend_params=ByonoyLuminescence96Backend.LuminescenceParams(integration_time=5),
)

## Resource layout

The Luminescence 96 has an interlock: you cannot assign a plate to the base while the reader unit is on top. In an automated workcell, use a robotic arm to move the reader unit off the base before loading the plate.

```
ByonoyLuminescenceBaseUnit (base)
 +-- plate_holder         (assign plates here)
 +-- reader_unit_holder   (reader unit sits here during measurement)
```

## Teardown

In [ ]:
await reader.stop()